# Table 2 — Light-deflection corrections for planetary satellites

Estimate $C_E$ for nine satellites in two representative configurations:
well separated from the host planet (`CE_sep`) and near its limb (`CE_limb`).
The final table gives the adopted orbital radius, the observer-to-planet
distance interval, and the corresponding correction ranges.

The model uses circular orbits in the plane containing the observer and planet,
spherical bodies, and a star at infinity aligned with the satellite's geometric
direction. Only the host body's monopole deflection is included. The equations
and interpretation of $C_E$ are given in the accompanying article.

Run the cells in order with **NumPy, Astropy, and `light_deflection.py`** available.
Place that module beside this notebook, or install it on the Python path.
`calc_alpha_star` must return an angular quantity, such as mas. No ephemeris
kernels or observing epoch are required.

In [1]:
import numpy as np
import astropy.units as u
import astropy.constants as const
from astropy.coordinates import CartesianRepresentation
from astropy.table import QTable, Table

from light_deflection import delta_kpn, calc_alpha_star, calc_CE

## 1. Adopted parameters

Masses, spherical radii, orbital radii, and distance bounds are fixed inputs for
these estimates. Satellite semimajor axes are treated as circular orbital radii
$a$; eccentricity and inclination are not modeled. Compute $GM$ from the adopted
mass using Astropy's gravitational constant.

The distance bounds for Mars through Neptune use a representative heliocentric
distance minus/plus 1 au. Pluto uses 29.66 − 1 and 49.30 + 1 au. These are
illustrative bounds, not ephemeris-derived extrema. The heading “Planet” also
includes Pluto, a dwarf planet.

Parameter references: [satellite physical parameters](https://ssd.jpl.nasa.gov/sats/phys_par/sep.html),
[satellite mean elements](https://ssd.jpl.nasa.gov/sats/elem/sep.html),
[planetary physical parameters](https://ssd.jpl.nasa.gov/planets/phys_par.html), and
[approximate planetary positions](https://ssd.jpl.nasa.gov/planets/approx_pos.html).
The explicit values below define this calculation; executing the notebook does
not retrieve or update them.

In [2]:
planets = QTable(
    rows=[
        ("Mars",    0.641691e24,  3389.50,  0.52,  2.52),
        ("Jupiter", 1898.125e24, 69911.00,  4.20,  6.20),
        ("Saturn",   568.317e24, 58232.00,  8.54, 10.54),
        ("Uranus",   86.8099e24, 25362.00, 18.19, 20.19),
        ("Neptune", 102.4092e24, 24622.00, 29.07, 31.07),
        ("Pluto",    13024.6e18,  1188.30, 28.66, 50.30),
    ],
    names=["Planet", "mass", "radius", "D_A_min", "D_A_max"],
    units=[None, u.kg, u.km, u.au, u.au],
)
planets["GM"] = (const.G * planets["mass"]).to(u.km**3 / u.s**2)

satellites = QTable(
    rows=[
        ("Mars",    "Phobos",       9_375,   11.08),
        ("Jupiter", "Io",         421_800, 1821.49),
        ("Jupiter", "Europa",     671_100, 1560.80),
        ("Jupiter", "Ganymede", 1_070_400, 2631.20),
        ("Jupiter", "Callisto", 1_882_700, 2410.30),
        ("Saturn",  "Titan",    1_221_900, 2574.76),
        ("Uranus",  "Titania",    436_298,  788.90),
        ("Neptune", "Triton",     354_800, 1352.60),
        ("Pluto",   "Charon",      19_600,  606.00),
    ],
    names=["Planet", "Satellite", "orbit_radius", "radius"],
    units=[None, None, u.km, u.km],
)

## 2. Geometry and correction

Place the observer at $(0,0,0)$ and the planet at $(D_A,0,0)$. The satellite's
position relative to the planet is:

- **Separated:** $(0,a,0)$, representing the largest transverse orbital offset.
- **Near limb:** $(-\sqrt{a^2-b^2},b,0)$, where $b=R_A+R_{\rm satellite}$.
  This selects the foreground side of the orbit and places the satellite's disk
  next to the planet's limb in the approximation $D_A\gg a$. It is an estimate
  of external contact, not an exact apparent-contact solution.

The satellite radius only sets this limb configuration. Both configurations
evaluate the correction at the satellite center, and both offsets have norm $a$.
The negative orbital $x$ coordinate is important: the far-side limb is a different
configuration and is not included here.

For each configuration, obtain the star direction from the satellite position.
Use `delta_kpn` for the finite-distance satellite, `calc_alpha_star` for the star
at infinity, and `calc_CE` with **$D_E$ equal to the observer-to-satellite distance**.
Convert the stellar deflection from mas to numerical radians before constructing
its dimensionless vector.

In [3]:
def satellite_corrections(gm, D_A, planet_radius, orbit_radius, satellite_radius):
    """Return [CE_sep, CE_limb] as a two-element Quantity in km.

    Parameters are scalar Astropy quantities: gm has units of length**3/time**2;
    D_A and all radii have length units. The orbit must lie outside the planet,
    and the observer must lie outside the satellite orbit. The limb geometry
    uses approximate foreground external contact; the star is at infinity.
    """
    D_A = u.Quantity(D_A).to(u.km)
    a = u.Quantity(orbit_radius).to(u.km)
    b = (planet_radius + satellite_radius).to(u.km)
    if not (D_A > a > b > 0 * u.km):
        raise ValueError("Require D_A > orbit_radius > sum of radii > 0.")

    planet = CartesianRepresentation(D_A, 0 * u.km, 0 * u.km)
    # Array order: separated configuration, foreground limb configuration.
    offset = CartesianRepresentation(
        u.Quantity([0 * u.km, -np.sqrt(a**2 - b**2)]),
        u.Quantity([a, b]),
        np.zeros(2) * u.km,
    )
    target = planet + offset
    D_E = target.norm()
    psi = np.arctan2(target.y, target.x)

    delta_target = delta_kpn(
        gm, vec_rso=-target, vec_rao=-planet, vec_ras=offset
    )
    alpha_star = calc_alpha_star(gm, D_A, psi).to_value(
        u.rad, equivalencies=u.dimensionless_angles()
    )
    delta_star = CartesianRepresentation(
        np.sin(psi), -np.cos(psi), np.zeros(2)
    ) * alpha_star

    return calc_CE(D_E, delta_target, delta_star).to(u.km)

## 3. Evaluate the distance endpoints

For each satellite, evaluate both configurations at $D_{A,\min}$ and
$D_{A,\max}$. The reported $C_E$ ranges are the smaller and larger values of
those two evaluations. They do not represent a search over orbital phase or
actual observing dates. `results` retains unrounded values with physical units.

In [4]:
rows = []
for satellite in satellites:
    planet = planets[planets["Planet"] == satellite["Planet"]][0]
    endpoint_corrections = u.Quantity([
        satellite_corrections(
            planet["GM"], distance, planet["radius"],
            satellite["orbit_radius"], satellite["radius"],
        )
        for distance in [planet["D_A_min"], planet["D_A_max"]]
    ])
    CE_sep = endpoint_corrections[:, 0]
    CE_limb = endpoint_corrections[:, 1]
    rows.append((
        satellite["Planet"], satellite["Satellite"], satellite["orbit_radius"],
        planet["D_A_min"], planet["D_A_max"],
        CE_sep.min(), CE_sep.max(), CE_limb.min(), CE_limb.max(),
    ))

results = QTable(
    rows=rows,
    names=[
        "Planet", "Satellite", "orbit_radius", "D_A_min", "D_A_max",
        "CE_sep_min", "CE_sep_max", "CE_limb_min", "CE_limb_max",
    ],
)
for column in ["CE_sep_min", "CE_sep_max", "CE_limb_min", "CE_limb_max"]:
    if not np.all(np.isfinite(results[column]) & (results[column] > 0 * u.km)):
        raise RuntimeError(f"Invalid correction in {column}.")

## 4. Table 2

Distances $D_A$ are in au; orbital radii and corrections are in km.
Corrections are displayed with three significant figures. Each “min–max” entry
is an interval, not a subtraction. Full precision remains available in `results`.
Set `export_tables = True` to write the numerical table (ECSV, including units)
and the formatted table (CSV) in the working directory; existing files with
these names are replaced.

In [5]:
def format_range(lower, upper, unit, spec):
    """Format two quantities in a common unit as a min–max interval."""
    return f"{lower.to_value(unit):{spec}} – {upper.to_value(unit):{spec}}"


table2 = Table(
    rows=[
        (
            row["Planet"], row["Satellite"], f"{row['orbit_radius'].to_value(u.km):,.0f}",
            format_range(row["D_A_min"], row["D_A_max"], u.au, ".2f"),
            format_range(row["CE_sep_min"], row["CE_sep_max"], u.km, ".3g"),
            format_range(row["CE_limb_min"], row["CE_limb_max"], u.km, ".3g"),
        )
        for row in results
    ],
    names=[
        "Planet", "Satellite", "Orbit radius [km]", "D_A (min–max) [au]",
        "CE_sep (min–max) [km]", "CE_limb (min–max) [km]",
    ],
)

export_tables = False
if export_tables:
    results.write("Table2_results.ecsv", format="ascii.ecsv", overwrite=True)
    table2.write("Table2_display.csv", format="ascii.csv", overwrite=True)

table2

Planet,Satellite,Orbit radius [km],D_A (min–max) [au],CE_sep (min–max) [km],CE_limb (min–max) [km]
str7,str8,str9,str13,str16,str14
Mars,Phobos,"9,375",0.52 – 2.52,0.0158 – 0.0766,0.0436 – 0.211
Jupiter,Io,"421,800",4.20 – 6.20,8.4 – 12.4,49.3 – 72.8
Jupiter,Europa,"671,100",4.20 – 6.20,5.28 – 7.79,49.5 – 73.1
Jupiter,Ganymede,"1,070,400",4.20 – 6.20,3.31 – 4.88,48.7 – 71.9
Jupiter,Callisto,"1,882,700",4.20 – 6.20,1.88 – 2.77,48.7 – 72
Saturn,Titan,"1,221,900",8.54 – 10.54,1.76 – 2.18,35.4 – 43.7
Uranus,Titania,"436,298",18.19 – 20.19,1.61 – 1.79,26.8 – 29.8
Neptune,Triton,"354,800",29.07 – 31.07,3.73 – 3.99,50.9 – 54.4
Pluto,Charon,"19,600",28.66 – 50.30,0.00846 – 0.0149,0.0924 – 0.162
